<a href="https://colab.research.google.com/github/postnicov/RowingSPb/blob/main/Rowing_EMD_Analysis_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1. Install required packages and import libraries

import importlib
import subprocess
import sys

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Core packages
_ensure("EMD-signal", "PyEMD")
_ensure("plotly")
_ensure("pandas")
_ensure("numpy")
_ensure("scipy")

# Plotly 5.24.1 is compatible with kaleido 0.2.1
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaleido==0.2.1"])

# Imports
import numpy as np
import pandas as pd
import re
from collections import defaultdict

import plotly
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"

#import kaleido
from scipy.signal import hilbert
from PyEMD import EMD

In [ ]:
#@title 2. Graphical settings (font, size, line width)
# ── Global plot style variables ───────────────────────────────────────────────
FONT_FAMILY  = "serif"
FONT_SIZE    = 14
LINE_WIDTH   = 1

LAYOUT_DEFAULTS = dict(
    font=dict(family=FONT_FAMILY, size=FONT_SIZE),
    legend=dict(font=dict(family=FONT_FAMILY, size=FONT_SIZE - 1)),
    xaxis=dict(title_font=dict(family=FONT_FAMILY, size=FONT_SIZE),
               tickfont=dict(family=FONT_FAMILY, size=FONT_SIZE - 1)),
    yaxis=dict(title_font=dict(family=FONT_FAMILY, size=FONT_SIZE),
               tickfont=dict(family=FONT_FAMILY, size=FONT_SIZE - 1)),
)

print(f"Font: {FONT_FAMILY}, Size: {FONT_SIZE}, Line width: {LINE_WIDTH}")

In [ ]:
#@title 3. Retrieve data from GitHub

import time
import requests
from io import StringIO

DATA_URL = (
    "https://raw.githubusercontent.com/postnicov/RowingSPb/"
    "refs/heads/main/data/2026_05_27_1_MH8%2BTest.csv"
)

headers = {
    "User-Agent": "Mozilla/5.0"
}

max_tries = 5
wait = 2

for attempt in range(max_tries):
    response = requests.get(DATA_URL, headers=headers, timeout=60)
    if response.status_code == 200:
        break
    elif response.status_code == 429 and attempt < max_tries - 1:
        time.sleep(wait)
        wait *= 2
    else:
        response.raise_for_status()

df_raw = pd.read_csv(StringIO(response.text))

print(f"Loaded {len(df_raw)} rows × {len(df_raw.columns)} columns")
print("Columns:", list(df_raw.columns))
df_raw.head(3)

In [ ]:
#@title 4. Column descriptions
from IPython.display import Markdown, display

display(Markdown("""
## Column descriptions

- **T** — time, s
- **As** — boat acceleration, m/s²
- **Vs** — boat speed (GPS), m/s
- **GPSd** — distance covered (GPS), m
- **A#** — horizontal oar angle, degrees (# = rower number counted from the stern; stroke = 1)
- **V#** — vertical oar angle, degrees
- **H#** — handle force, N
- **S#** — seat position, m
"""))

In [ ]:
#@title 5. Plot As vs T (interactive Plotly)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_raw["T"], y=df_raw["As"],
    mode="lines",
    name="As",
    line=dict(width=LINE_WIDTH),
))

fig.update_layout(
    title="Raw data — As vs T",
    xaxis_title="T (s)",
    yaxis_title="As",
    **LAYOUT_DEFAULTS,
)
fig.show()

In [ ]:
#@title 6. Set time window: Tmin and Tmax in the original data
Tmin = float(df_raw["T"].min())
Tmax = float(df_raw["T"].max())

print(f"Tmin = {Tmin:.4f} s")
print(f"Tmax = {Tmax:.4f} s")

In [ ]:
#@title 6a. Retype Tmin and Tmax to the range of interest
Tmin = 1.92  #@param {type:"number"}
Tmax = 384  #@param {type:"number"}

print(f"Tmin = {Tmin:.4f} s")
print(f"Tmax = {Tmax:.4f} s")

In [ ]:
#@title 7. Trim to [Tmin, Tmax] and redefine T = T - Tmin
mask = (df_raw["T"] >= Tmin) & (df_raw["T"] <= Tmax)
df = df_raw.loc[mask].copy().reset_index(drop=True)
df["T"] = df["T"] - Tmin

print(f"Rows after trimming: {len(df)}")
print(f"New T range: [{df['T'].min():.4f}, {df['T'].max():.4f}] s")

In [ ]:
#@title 8. Empirical Mode Decomposition for each signal column
signal_cols = [c for c in df.columns if c not in ("N", "T")]

T_arr  = df["T"].to_numpy()
emd    = EMD()
imfs   = {}
trends = {}

for col in signal_cols:
    sig = df[col].to_numpy(dtype=float)
    result = emd.emd(sig, T_arr)
    imfs[col]   = result
    trends[col] = result[-1]
    print(f"  {col:5s}: {result.shape[0]} IMFs decomposed")

print("EMD complete.")

In [ ]:
#@title 9. Plot maximal trend IMF grouped by letter prefix
groups = defaultdict(list)
for col in signal_cols:
    m = re.match(r'^([A-Za-z]+)', col)
    if m:
        groups[m.group(1)].append(col)

for letter, members in sorted(groups.items()):
    if len(members) <= 1:
        continue
    fig = go.Figure()
    for col in sorted(members):
        fig.add_trace(go.Scatter(
            x=T_arr, y=trends[col],
            mode="lines",
            name=col,
            line=dict(width=LINE_WIDTH),
        ))
    fig.update_layout(
        title=f"Trend IMF — group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title=f"{letter} (trend IMF)",
        **LAYOUT_DEFAULTS,
    )
    fig.show()

In [ ]:
#@title 10. Detrend signals by subtracting the trend IMF
detrended = {}
for col in signal_cols:
    detrended[col] = df[col].to_numpy(dtype=float) - trends[col]

print("Detrending complete.")
for col in signal_cols[:3]:
    print(f"  {col}: mean detrended = {detrended[col].mean():.4f}")

In [ ]:
#@title 11. Plot detrended signals only for grouped variables
grouped_only = {
    letter: sorted(members)
    for letter, members in groups.items()
    if len(members) > 1
}

for letter, members in sorted(grouped_only.items()):
    fig = go.Figure()
    for col in members:
        fig.add_trace(go.Scatter(
            x=T_arr,
            y=detrended[col],
            mode="lines",
            name=col,
            line=dict(width=LINE_WIDTH),
        ))
    fig.update_layout(
        title=f"Detrended signals — group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title=f"{letter} (detrended)",
        **LAYOUT_DEFAULTS,
    )
    fig.show()

In [ ]:
#@title 12. Hilbert transforms and Hilbert phases for grouped variables

# Install SciPy if needed
_ensure("scipy")

from scipy.signal import hilbert

hilbert_signals = {}
hilbert_phases = {}

grouped_only = {
    letter: sorted(members)
    for letter, members in groups.items()
    if len(members) > 1
}

for letter, members in sorted(grouped_only.items()):
    hilbert_signals[letter] = {}
    hilbert_phases[letter] = {}

    for col in members:
        x = np.asarray(detrended[col], dtype=float)
        analytic_signal = hilbert(x)
        phase = np.unwrap(np.angle(analytic_signal))

        hilbert_signals[letter][col] = analytic_signal
        hilbert_phases[letter][col] = phase

    print(f"{letter}: Hilbert transform and phase computed for {len(members)} variables")

In [ ]:
#@title 13. Hilbert phase difference respectively to stroke rower

for letter, members in sorted(grouped_only.items()):
    if "1" not in "".join(members):
        print(f"Group {letter}: no stroke-rower channel found, skipped")
        continue

    ref_col = f"{letter}1"
    if ref_col not in members:
        print(f"Group {letter}: {ref_col} not found, skipped")
        continue

    ref_phase = hilbert_phases[letter][ref_col]

    fig = go.Figure()
    for col in members:
        if col == ref_col:
            continue
        phase_diff = hilbert_phases[letter][col] - ref_phase

        fig.add_trace(go.Scatter(
            x=T_arr,
            y=phase_diff,
            mode="lines",
            name=f"{col} - {ref_col}",
            line=dict(width=LINE_WIDTH),
        ))

    fig.update_layout(
        title=f"Hilbert phase difference respectively to stroke rower — group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title="Phase difference (rad)",
        **LAYOUT_DEFAULTS,
    )
    fig.show()

In [ ]:
#@title 13a. Hilbert phase difference respectively to stroke rower (wrapped respectively to $2\pi$)

def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

for letter, members in sorted(grouped_only.items()):
    ref_col = f"{letter}1"
    if ref_col not in members:
        print(f"Group {letter}: {ref_col} not found, skipped")
        continue

    ref_phase = hilbert_phases[letter][ref_col]

    fig = go.Figure()
    for col in members:
        if col == ref_col:
            continue

        raw_phase_diff = hilbert_phases[letter][col] - ref_phase
        phase_diff = wrap_to_pi(raw_phase_diff)

        fig.add_trace(go.Scatter(
            x=T_arr,
            y=phase_diff,
            mode="lines",
            name=f"{col} - {ref_col}",
            line=dict(width=LINE_WIDTH),
        ))

    fig.update_layout(
        title=f"Hilbert phase difference respectively to stroke rower — group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title="Wrapped phase difference (rad)",
        **LAYOUT_DEFAULTS,
    )
    fig.show()

In [ ]:
#@title 14. Hilbert phase difference between subsequent rowers

import re

def rower_number(colname):
    m = re.search(r'(\d+)$', colname)
    return int(m.group(1)) if m else None

for letter, members in sorted(grouped_only.items()):
    numbered_members = [col for col in members if rower_number(col) is not None]
    ordered_members = sorted(numbered_members, key=rower_number)

    if len(ordered_members) < 2:
        print(f"Group {letter}: fewer than two numbered rowers, skipped")
        continue

    fig = go.Figure()
    for i in range(1, len(ordered_members)):
        prev_col = ordered_members[i - 1]
        curr_col = ordered_members[i]

        phase_diff = hilbert_phases[letter][curr_col] - hilbert_phases[letter][prev_col]

        fig.add_trace(go.Scatter(
            x=T_arr,
            y=phase_diff,
            mode="lines",
            name=f"{curr_col} - {prev_col}",
            line=dict(width=LINE_WIDTH),
        ))

    fig.update_layout(
        title=f"Hilbert phase difference between subsequent rowers — group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title="Phase difference (rad)",
        **LAYOUT_DEFAULTS,
    )
    fig.show()

In [ ]:
#@title 14. Hilbert phase difference between subsequent rowers (wrapped respectively to $2\pi$)

import re

def rower_number(colname):
    m = re.search(r'(\d+)$', colname)
    return int(m.group(1)) if m else None

def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

for letter, members in sorted(grouped_only.items()):
    numbered_members = [col for col in members if rower_number(col) is not None]
    ordered_members = sorted(numbered_members, key=rower_number)

    if len(ordered_members) < 2:
        print(f"Group {letter}: fewer than two numbered rowers, skipped")
        continue

    fig = go.Figure()
    for i in range(1, len(ordered_members)):
        prev_col = ordered_members[i - 1]
        curr_col = ordered_members[i]

        raw_phase_diff = hilbert_phases[letter][curr_col] - hilbert_phases[letter][prev_col]
        phase_diff = wrap_to_pi(raw_phase_diff)

        fig.add_trace(go.Scatter(
            x=T_arr,
            y=phase_diff,
            mode="lines",
            name=f"{curr_col} - {prev_col}",
            line=dict(width=LINE_WIDTH),
        ))

    fig.update_layout(
        title=f"Hilbert phase difference between subsequent rowers — group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title="Wrapped phase difference (rad)",
        **LAYOUT_DEFAULTS,
    )
    fig.show()

In [ ]:
#@title 15. Export all figures to HTML and pack into ZIP

import re
import zipfile
from pathlib import Path
from google.colab import files

EXPORT_DIR = Path("exported_figures_html")
EXPORT_DIR.mkdir(exist_ok=True)

def safe_filename(text):
    text = str(text).strip()
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace("—", "-").replace("–", "-")
    text = re.sub(r'[\\/:*?"<>|]+', "_", text)
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text[:180].strip("._-")

def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

def rower_number(colname):
    m = re.search(r'(\d+)$', colname)
    return int(m.group(1)) if m else None

all_figures = []

# Figure from Cell 5
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_raw["T"],
    y=df_raw["As"],
    mode="lines",
    name="As",
    line=dict(width=LINE_WIDTH),
))
fig.update_layout(
    title="Raw data - As vs T",
    xaxis_title="T (s)",
    yaxis_title="As",
    **LAYOUT_DEFAULTS,
)
all_figures.append(fig)

# Figures from Cell 9
for letter, members in sorted(groups.items()):
    if len(members) <= 1:
        continue

    fig = go.Figure()
    for col in sorted(members):
        fig.add_trace(go.Scatter(
            x=T_arr,
            y=trends[col],
            mode="lines",
            name=col,
            line=dict(width=LINE_WIDTH),
        ))

    fig.update_layout(
        title=f"Trend IMF - group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title=f"{letter} (trend IMF)",
        **LAYOUT_DEFAULTS,
    )
    all_figures.append(fig)

# Figures from Cell 11
for letter, members in sorted(grouped_only.items()):
    fig = go.Figure()
    for col in members:
        fig.add_trace(go.Scatter(
            x=T_arr,
            y=detrended[col],
            mode="lines",
            name=col,
            line=dict(width=LINE_WIDTH),
        ))

    fig.update_layout(
        title=f"Detrended signals - group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title=f"{letter} (detrended)",
        **LAYOUT_DEFAULTS,
    )
    all_figures.append(fig)

# Figures from Cell 13
for letter, members in sorted(grouped_only.items()):
    ref_col = f"{letter}1"
    if ref_col not in members:
        continue

    ref_phase = hilbert_phases[letter][ref_col]
    fig = go.Figure()

    for col in members:
        if col == ref_col:
            continue

        raw_phase_diff = hilbert_phases[letter][col] - ref_phase
        phase_diff = wrap_to_pi(raw_phase_diff)

        fig.add_trace(go.Scatter(
            x=T_arr,
            y=phase_diff,
            mode="lines",
            name=f"{col} - {ref_col}",
            line=dict(width=LINE_WIDTH),
        ))

    fig.update_layout(
        title=f"Hilbert phase difference respectively to stroke rower - group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title="Wrapped phase difference (rad)",
        **LAYOUT_DEFAULTS,
    )
    all_figures.append(fig)

# Figures from Cell 14
for letter, members in sorted(grouped_only.items()):
    numbered_members = [col for col in members if rower_number(col) is not None]
    ordered_members = sorted(numbered_members, key=rower_number)

    if len(ordered_members) < 2:
        continue

    fig = go.Figure()
    for i in range(1, len(ordered_members)):
        prev_col = ordered_members[i - 1]
        curr_col = ordered_members[i]

        raw_phase_diff = hilbert_phases[letter][curr_col] - hilbert_phases[letter][prev_col]
        phase_diff = wrap_to_pi(raw_phase_diff)

        fig.add_trace(go.Scatter(
            x=T_arr,
            y=phase_diff,
            mode="lines",
            name=f"{curr_col} - {prev_col}",
            line=dict(width=LINE_WIDTH),
        ))

    fig.update_layout(
        title=f"Hilbert phase difference between subsequent rowers - group '{letter}'",
        xaxis_title="T (s)",
        yaxis_title="Wrapped phase difference (rad)",
        **LAYOUT_DEFAULTS,
    )
    all_figures.append(fig)

saved_files = []

for i, fig in enumerate(all_figures, start=1):
    title = fig.layout.title.text if fig.layout.title.text else f"figure_{i}"
    filename = safe_filename(title) + ".html"
    filepath = EXPORT_DIR / filename

    fig.write_html(str(filepath), include_plotlyjs="cdn", full_html=True)
    saved_files.append(filepath)

zip_name = "rowing_figures_html.zip"

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in saved_files:
        zf.write(file_path, arcname=file_path.name)

print(f"Saved {len(saved_files)} HTML files into {zip_name}")
files.download(zip_name)

In [ ]:
#@title 16a. Restrict plotted time interval for Cell 16

Tbegin = 0  #@param {type:"number"}
Tend   = 384  #@param {type:"number"}

plot_mask = (T_arr >= Tbegin) & (T_arr <= Tend)

T_plot = T_arr[plot_mask]

trends_plot = {
    col: np.asarray(trends[col])[plot_mask]
    for col in trends
}

detrended_plot = {
    col: np.asarray(detrended[col])[plot_mask]
    for col in detrended
}

hilbert_phases_plot = {
    letter: {
        col: np.asarray(hilbert_phases[letter][col])[plot_mask]
        for col in hilbert_phases[letter]
    }
    for letter in hilbert_phases
}

print(f"Restricted plotting interval: {Tbegin:.3f} to {Tend:.3f} s")
print(f"Number of points kept: {plot_mask.sum()}")

In [ ]:
#@title 16. Static Matplotlib export of all figure sets to zipped PNG

import re
import zipfile
from pathlib import Path
from math import ceil

import matplotlib.pyplot as plt
from google.colab import files

# Fallback: if Cell 16a was not run, use full arrays
if "T_plot" not in globals():
    T_plot = T_arr
    trends_plot = {col: np.asarray(trends[col]) for col in trends}
    detrended_plot = {col: np.asarray(detrended[col]) for col in detrended}
    hilbert_phases_plot = {
        letter: {col: np.asarray(hilbert_phases[letter][col]) for col in hilbert_phases[letter]}
        for letter in hilbert_phases
    }

EXPORT_DIR = Path("exported_figures_matplotlib")
EXPORT_DIR.mkdir(exist_ok=True)

def safe_filename(text):
    text = str(text).strip()
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace("—", "-").replace("–", "-")
    text = re.sub(r'[\\/:*?"<>|]+', "_", text)
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text[:180].strip("._-")

def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

def rower_number(colname):
    m = re.search(r'(\d+)$', colname)
    return int(m.group(1)) if m else None

plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["font.size"] = FONT_SIZE
plt.rcParams["axes.titlesize"] = FONT_SIZE
plt.rcParams["axes.labelsize"] = FONT_SIZE
plt.rcParams["legend.fontsize"] = FONT_SIZE - 1
plt.rcParams["xtick.labelsize"] = FONT_SIZE - 1
plt.rcParams["ytick.labelsize"] = FONT_SIZE - 1

saved_files = []

# ================================================================
# Figure set 1: Raw As vs T in restricted interval
# ================================================================
title = "Raw data - As vs T"
fig, ax = plt.subplots(figsize=(10, 5), dpi=200)

raw_mask = (df_raw["T"] >= T_plot.min()) & (df_raw["T"] <= T_plot.max())
ax.plot(df_raw.loc[raw_mask, "T"], df_raw.loc[raw_mask, "As"], linewidth=LINE_WIDTH, label="As")

ax.set_title(title)
ax.set_xlabel("T (s)")
ax.set_ylabel("As")
ax.grid(True, alpha=0.3)
ax.legend()

filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
fig.tight_layout()
fig.savefig(filepath, dpi=200, bbox_inches="tight")
saved_files.append(filepath)
plt.close(fig)

# ================================================================
# Figure set 2: Trend IMF grouped by letter
# ================================================================
trend_groups = [(letter, members) for letter, members in sorted(groups.items()) if len(members) > 1]

if len(trend_groups) > 0:
    ncols = 2
    nrows = ceil(len(trend_groups) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), dpi=200, squeeze=False)
    axes = axes.flatten()

    for ax, (letter, members) in zip(axes, trend_groups):
        for col in sorted(members):
            ax.plot(T_plot, trends_plot[col], linewidth=LINE_WIDTH, label=col)
        ax.set_title(f"Trend IMF - group '{letter}'")
        ax.set_xlabel("T (s)")
        ax.set_ylabel(f"{letter} (trend IMF)")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)

    for ax in axes[len(trend_groups):]:
        ax.axis("off")

    title = "Trend_IMF_grouped_subplots"
    filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
    fig.tight_layout()
    fig.savefig(filepath, dpi=200, bbox_inches="tight")
    saved_files.append(filepath)
    plt.close(fig)

# ================================================================
# Figure set 3: Detrended grouped variables
# ================================================================
detrended_groups = [(letter, members) for letter, members in sorted(grouped_only.items()) if len(members) > 1]

if len(detrended_groups) > 0:
    ncols = 2
    nrows = ceil(len(detrended_groups) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), dpi=200, squeeze=False)
    axes = axes.flatten()

    for ax, (letter, members) in zip(axes, detrended_groups):
        for col in sorted(members):
            ax.plot(T_plot, detrended_plot[col], linewidth=LINE_WIDTH, label=col)
        ax.set_title(f"Detrended signals - group '{letter}'")
        ax.set_xlabel("T (s)")
        ax.set_ylabel(f"{letter} (detrended)")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)

    for ax in axes[len(detrended_groups):]:
        ax.axis("off")

    title = "Detrended_signals_grouped_subplots"
    filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
    fig.tight_layout()
    fig.savefig(filepath, dpi=200, bbox_inches="tight")
    saved_files.append(filepath)
    plt.close(fig)

# ================================================================
# Figure set 4: Hilbert phase difference to stroke rower
# ================================================================
phase_ref_groups = []
for letter, members in sorted(grouped_only.items()):
    ref_col = f"{letter}1"
    if ref_col in members:
        phase_ref_groups.append((letter, members, ref_col))

if len(phase_ref_groups) > 0:
    ncols = 2
    nrows = ceil(len(phase_ref_groups) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), dpi=200, squeeze=False)
    axes = axes.flatten()

    for ax, (letter, members, ref_col) in zip(axes, phase_ref_groups):
        ref_phase = hilbert_phases_plot[letter][ref_col]
        for col in members:
            if col == ref_col:
                continue
            raw_phase_diff = hilbert_phases_plot[letter][col] - ref_phase
            phase_diff = wrap_to_pi(raw_phase_diff)
            ax.plot(T_plot, phase_diff, linewidth=LINE_WIDTH, label=f"{col} - {ref_col}")

        ax.set_title(f"Hilbert phase difference respectively to stroke rower - group '{letter}'")
        ax.set_xlabel("T (s)")
        ax.set_ylabel("Wrapped phase difference (rad)")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)

    for ax in axes[len(phase_ref_groups):]:
        ax.axis("off")

    title = "Hilbert_phase_difference_to_stroke_rower_grouped_subplots"
    filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
    fig.tight_layout()
    fig.savefig(filepath, dpi=200, bbox_inches="tight")
    saved_files.append(filepath)
    plt.close(fig)

# ================================================================
# Figure set 5: Hilbert phase difference between subsequent rowers
# ================================================================
phase_next_groups = []
for letter, members in sorted(grouped_only.items()):
    numbered_members = [col for col in members if rower_number(col) is not None]
    ordered_members = sorted(numbered_members, key=rower_number)
    if len(ordered_members) >= 2:
        phase_next_groups.append((letter, ordered_members))

if len(phase_next_groups) > 0:
    ncols = 2
    nrows = ceil(len(phase_next_groups) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), dpi=200, squeeze=False)
    axes = axes.flatten()

    for ax, (letter, ordered_members) in zip(axes, phase_next_groups):
        for i in range(1, len(ordered_members)):
            prev_col = ordered_members[i - 1]
            curr_col = ordered_members[i]
            raw_phase_diff = hilbert_phases_plot[letter][curr_col] - hilbert_phases_plot[letter][prev_col]
            phase_diff = wrap_to_pi(raw_phase_diff)
            ax.plot(T_plot, phase_diff, linewidth=LINE_WIDTH, label=f"{curr_col} - {prev_col}")

        ax.set_title(f"Hilbert phase difference between subsequent rowers - group '{letter}'")
        ax.set_xlabel("T (s)")
        ax.set_ylabel("Wrapped phase difference (rad)")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)

    for ax in axes[len(phase_next_groups):]:
        ax.axis("off")

    title = "Hilbert_phase_difference_between_subsequent_rowers_grouped_subplots"
    filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
    fig.tight_layout()
    fig.savefig(filepath, dpi=200, bbox_inches="tight")
    saved_files.append(filepath)
    plt.close(fig)

zip_name = "rowing_figures_matplotlib_png.zip"

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in saved_files:
        zf.write(file_path, arcname=file_path.name)

print(f"Saved {len(saved_files)} Matplotlib PNG figures into {zip_name}")
print(f"Displayed/exported interval: {T_plot.min():.3f} to {T_plot.max():.3f} s")
files.download(zip_name)

In [ ]:
#@title 16. Static Matplotlib export of all figure sets to zipped PNG

import re
import zipfile
from pathlib import Path
from math import ceil

import matplotlib.pyplot as plt
from google.colab import files

# Fallback: if Cell 16a was not run, use full arrays
if "T_plot" not in globals():
    T_plot = T_arr
    trends_plot = {col: np.asarray(trends[col]) for col in trends}
    detrended_plot = {col: np.asarray(detrended[col]) for col in detrended}
    hilbert_phases_plot = {
        letter: {col: np.asarray(hilbert_phases[letter][col]) for col in hilbert_phases[letter]}
        for letter in hilbert_phases
    }

EXPORT_DIR = Path("exported_figures_matplotlib")
EXPORT_DIR.mkdir(exist_ok=True)

def safe_filename(text):
    text = str(text).strip()
    text = re.sub(r"<[^>]+>", "", text)
    text = text.replace("—", "-").replace("–", "-")
    text = re.sub(r'[\\/:*?"<>|]+', "_", text)
    text = re.sub(r"\s+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text[:180].strip("._-")

def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

def rower_number(colname):
    m = re.search(r'(\d+)$', colname)
    return int(m.group(1)) if m else None

plt.rcParams["font.family"] = FONT_FAMILY
plt.rcParams["font.size"] = FONT_SIZE
plt.rcParams["axes.titlesize"] = FONT_SIZE
plt.rcParams["axes.labelsize"] = FONT_SIZE
plt.rcParams["legend.fontsize"] = FONT_SIZE - 1
plt.rcParams["xtick.labelsize"] = FONT_SIZE - 1
plt.rcParams["ytick.labelsize"] = FONT_SIZE - 1

saved_files = []

# ================================================================
# Figure set 1: Raw As vs T in restricted interval
# ================================================================
title = "Raw data - As vs T"
fig, ax = plt.subplots(figsize=(10, 5), dpi=200)

raw_mask = (df_raw["T"] >= T_plot.min()) & (df_raw["T"] <= T_plot.max())
ax.plot(df_raw.loc[raw_mask, "T"], df_raw.loc[raw_mask, "As"], linewidth=LINE_WIDTH, label="As")

ax.set_title(title)
ax.set_xlabel("T (s)")
ax.set_ylabel("As")
ax.grid(True, alpha=0.3)
ax.legend()

filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
fig.tight_layout()
fig.savefig(filepath, dpi=200, bbox_inches="tight")
saved_files.append(filepath)
plt.close(fig)

# ================================================================
# Figure set 2: Trend IMF grouped by letter
# ================================================================
trend_groups = [(letter, members) for letter, members in sorted(groups.items()) if len(members) > 1]

if len(trend_groups) > 0:
    ncols = 2
    nrows = ceil(len(trend_groups) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), dpi=200, squeeze=False)
    axes = axes.flatten()

    for ax, (letter, members) in zip(axes, trend_groups):
        for col in sorted(members):
            ax.plot(T_plot, trends_plot[col], linewidth=LINE_WIDTH, label=col)
        ax.set_title(f"Trend IMF - group '{letter}'")
        ax.set_xlabel("T (s)")
        ax.set_ylabel(f"{letter} (trend IMF)")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)

    for ax in axes[len(trend_groups):]:
        ax.axis("off")

    title = "Trend_IMF_grouped_subplots"
    filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
    fig.tight_layout()
    fig.savefig(filepath, dpi=200, bbox_inches="tight")
    saved_files.append(filepath)
    plt.close(fig)

# ================================================================
# Figure set 3: Detrended grouped variables
# ================================================================
detrended_groups = [(letter, members) for letter, members in sorted(grouped_only.items()) if len(members) > 1]

if len(detrended_groups) > 0:
    ncols = 2
    nrows = ceil(len(detrended_groups) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), dpi=200, squeeze=False)
    axes = axes.flatten()

    for ax, (letter, members) in zip(axes, detrended_groups):
        for col in sorted(members):
            ax.plot(T_plot, detrended_plot[col], linewidth=LINE_WIDTH, label=col)
        ax.set_title(f"Detrended signals - group '{letter}'")
        ax.set_xlabel("T (s)")
        ax.set_ylabel(f"{letter} (detrended)")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)

    for ax in axes[len(detrended_groups):]:
        ax.axis("off")

    title = "Detrended_signals_grouped_subplots"
    filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
    fig.tight_layout()
    fig.savefig(filepath, dpi=200, bbox_inches="tight")
    saved_files.append(filepath)
    plt.close(fig)

# ================================================================
# Figure set 4: Hilbert phase difference to stroke rower (UNWRAPPED)
# ================================================================
phase_ref_groups = []
for letter, members in sorted(grouped_only.items()):
    ref_col = f"{letter}1"
    if ref_col in members:
        phase_ref_groups.append((letter, members, ref_col))

if len(phase_ref_groups) > 0:
    ncols = 2
    nrows = ceil(len(phase_ref_groups) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), dpi=200, squeeze=False)
    axes = axes.flatten()

    for ax, (letter, members, ref_col) in zip(axes, phase_ref_groups):
        ref_phase = hilbert_phases_plot[letter][ref_col]
        for col in members:
            if col == ref_col:
                continue
            phase_diff = hilbert_phases_plot[letter][col] - ref_phase
            ax.plot(T_plot, phase_diff, linewidth=LINE_WIDTH, label=f"{col} - {ref_col}")

        ax.set_title(f"Hilbert phase difference respectively to stroke rower - group '{letter}' (unwrapped)")
        ax.set_xlabel("T (s)")
        ax.set_ylabel("Phase difference (rad)")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)

    for ax in axes[len(phase_ref_groups):]:
        ax.axis("off")

    title = "Hilbert_phase_difference_to_stroke_rower_unwrapped_grouped_subplots"
    filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
    fig.tight_layout()
    fig.savefig(filepath, dpi=200, bbox_inches="tight")
    saved_files.append(filepath)
    plt.close(fig)

# ================================================================
# Figure set 5: Hilbert phase difference between subsequent rowers (UNWRAPPED)
# ================================================================
phase_next_groups = []
for letter, members in sorted(grouped_only.items()):
    numbered_members = [col for col in members if rower_number(col) is not None]
    ordered_members = sorted(numbered_members, key=rower_number)
    if len(ordered_members) >= 2:
        phase_next_groups.append((letter, ordered_members))

if len(phase_next_groups) > 0:
    ncols = 2
    nrows = ceil(len(phase_next_groups) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), dpi=200, squeeze=False)
    axes = axes.flatten()

    for ax, (letter, ordered_members) in zip(axes, phase_next_groups):
        for i in range(1, len(ordered_members)):
            prev_col = ordered_members[i - 1]
            curr_col = ordered_members[i]
            phase_diff = hilbert_phases_plot[letter][curr_col] - hilbert_phases_plot[letter][prev_col]
            ax.plot(T_plot, phase_diff, linewidth=LINE_WIDTH, label=f"{curr_col} - {prev_col}")

        ax.set_title(f"Hilbert phase difference between subsequent rowers - group '{letter}' (unwrapped)")
        ax.set_xlabel("T (s)")
        ax.set_ylabel("Phase difference (rad)")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)

    for ax in axes[len(phase_next_groups):]:
        ax.axis("off")

    title = "Hilbert_phase_difference_between_subsequent_rowers_unwrapped_grouped_subplots"
    filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
    fig.tight_layout()
    fig.savefig(filepath, dpi=200, bbox_inches="tight")
    saved_files.append(filepath)
    plt.close(fig)

# ================================================================
# Figure set 6: Hilbert phase difference to stroke rower (WRAPPED)
# ================================================================
if len(phase_ref_groups) > 0:
    ncols = 2
    nrows = ceil(len(phase_ref_groups) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), dpi=200, squeeze=False)
    axes = axes.flatten()

    for ax, (letter, members, ref_col) in zip(axes, phase_ref_groups):
        ref_phase = hilbert_phases_plot[letter][ref_col]
        for col in members:
            if col == ref_col:
                continue
            raw_phase_diff = hilbert_phases_plot[letter][col] - ref_phase
            phase_diff = wrap_to_pi(raw_phase_diff)
            ax.plot(T_plot, phase_diff, linewidth=LINE_WIDTH, label=f"{col} - {ref_col}")

        ax.set_title(f"Hilbert phase difference respectively to stroke rower - group '{letter}' (wrapped)")
        ax.set_xlabel("T (s)")
        ax.set_ylabel("Wrapped phase difference (rad)")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)

    for ax in axes[len(phase_ref_groups):]:
        ax.axis("off")

    title = "Hilbert_phase_difference_to_stroke_rower_wrapped_grouped_subplots"
    filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
    fig.tight_layout()
    fig.savefig(filepath, dpi=200, bbox_inches="tight")
    saved_files.append(filepath)
    plt.close(fig)

# ================================================================
# Figure set 7: Hilbert phase difference between subsequent rowers (WRAPPED)
# ================================================================
if len(phase_next_groups) > 0:
    ncols = 2
    nrows = ceil(len(phase_next_groups) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows), dpi=200, squeeze=False)
    axes = axes.flatten()

    for ax, (letter, ordered_members) in zip(axes, phase_next_groups):
        for i in range(1, len(ordered_members)):
            prev_col = ordered_members[i - 1]
            curr_col = ordered_members[i]
            raw_phase_diff = hilbert_phases_plot[letter][curr_col] - hilbert_phases_plot[letter][prev_col]
            phase_diff = wrap_to_pi(raw_phase_diff)
            ax.plot(T_plot, phase_diff, linewidth=LINE_WIDTH, label=f"{curr_col} - {prev_col}")

        ax.set_title(f"Hilbert phase difference between subsequent rowers - group '{letter}' (wrapped)")
        ax.set_xlabel("T (s)")
        ax.set_ylabel("Wrapped phase difference (rad)")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2)

    for ax in axes[len(phase_next_groups):]:
        ax.axis("off")

    title = "Hilbert_phase_difference_between_subsequent_rowers_wrapped_grouped_subplots"
    filepath = EXPORT_DIR / f"{safe_filename(title)}.png"
    fig.tight_layout()
    fig.savefig(filepath, dpi=200, bbox_inches="tight")
    saved_files.append(filepath)
    plt.close(fig)

zip_name = "rowing_figures_matplotlib_png.zip"

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in saved_files:
        zf.write(file_path, arcname=file_path.name)

print(f"Saved {len(saved_files)} Matplotlib PNG figures into {zip_name}")
print(f"Displayed/exported interval: {T_plot.min():.3f} to {T_plot.max():.3f} s")
files.download(zip_name)